# Aula 15 — Inicialização de pesos: laboratório reproduzível

Este laboratório usa **NumPy puro** para testar quebra de simetria, Xavier/Glorot, He/Kaiming e propagação de escala. Não há autograd, downloads ou dados externos.

**Objetivos:** conferir variâncias teóricas, medir ativações e gradientes por camada e transformar a escolha da inicialização em um contrato executável.

> Execute as células em ordem. O notebook publicado permanece sem outputs; uma cópia é executada na validação.

## Dependências e reprodutibilidade

- Python ≥ 3.11
- NumPy ≥ 1.26
- Matplotlib ≥ 3.8
- nbformat ≥ 5.9 para validar o arquivo

Seed canônica: `20260915`. Cada experimento recebe um gerador explícito; repetir uma seed reproduz a sequência sem tornar os elementos de uma matriz iguais.

In [ ]:
import hashlib
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

warnings.filterwarnings("error")
SEED = 20260915

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)
assert tuple(map(int, np.__version__.split(".")[:2])) >= (1, 26)

## 1. Inicializadores explícitos

Na convenção `X @ W`, `W.shape == (fan_in, fan_out)`. A função devolve `float64` para tornar os diagnósticos numéricos mais precisos.

In [ ]:
def initialize(rng, fan_in, fan_out, scheme, *, alpha=0.0, dtype=np.float64):
    if fan_in <= 0 or fan_out <= 0:
        raise ValueError("fan_in e fan_out devem ser positivos")
    shape = (fan_in, fan_out)
    if scheme == "zeros":
        values = np.zeros(shape)
    elif scheme == "small_normal":
        values = rng.normal(0.0, 0.01, size=shape)
    elif scheme == "large_normal":
        values = rng.normal(0.0, 1.0, size=shape)
    elif scheme == "xavier_normal":
        std = np.sqrt(2.0 / (fan_in + fan_out))
        values = rng.normal(0.0, std, size=shape)
    elif scheme == "xavier_uniform":
        bound = np.sqrt(6.0 / (fan_in + fan_out))
        values = rng.uniform(-bound, bound, size=shape)
    elif scheme == "he_normal":
        gain = np.sqrt(2.0 / (1.0 + alpha**2))
        values = rng.normal(0.0, gain / np.sqrt(fan_in), size=shape)
    elif scheme == "he_uniform":
        gain = np.sqrt(2.0 / (1.0 + alpha**2))
        bound = np.sqrt(3.0) * gain / np.sqrt(fan_in)
        values = rng.uniform(-bound, bound, size=shape)
    else:
        raise ValueError(f"esquema desconhecido: {scheme}")
    return np.asarray(values, dtype=dtype)


for name in ("xavier_normal", "xavier_uniform", "he_normal", "he_uniform"):
    W = initialize(np.random.default_rng(SEED), 128, 64, name)
    print(f"{name:16s} shape={W.shape}, média={W.mean(): .6f}, variância={W.var():.6f}")

assert initialize(np.random.default_rng(SEED), 3, 2, "xavier_normal").shape == (3, 2)
try:
    initialize(np.random.default_rng(SEED), 0, 2, "he_normal")
except ValueError:
    pass
else:
    raise AssertionError("dimensão inválida não foi rejeitada")

## 2. Variâncias-alvo: normal e uniforme

Uma amostra grande permite verificar as fórmulas sem forçar a variância observada. A tolerância é estatística, não igualdade bit a bit.

In [ ]:
fan_in = fan_out = 1000
targets = {
    "xavier_normal": 2.0 / (fan_in + fan_out),
    "xavier_uniform": 2.0 / (fan_in + fan_out),
    "he_normal": 2.0 / fan_in,
    "he_uniform": 2.0 / fan_in,
}
observed = {}
for offset, (name, target) in enumerate(targets.items()):
    W = initialize(np.random.default_rng(SEED + offset), fan_in, fan_out, name)
    observed[name] = W.var()
    relative = abs(observed[name] - target) / target
    print(f"{name:16s} alvo={target:.8f}, observado={observed[name]:.8f}, erro rel.={relative:.4%}")
    assert relative < 0.01

assert abs(observed["xavier_normal"] - observed["xavier_uniform"]) < 3e-5
assert abs(observed["he_normal"] - observed["he_uniform"]) < 5e-5

## 3. Seed fixa não significa pesos iguais

Dois geradores novos com a mesma seed reproduzem a matriz. Dentro da matriz, os valores continuam distintos. Já duas chamadas sucessivas no mesmo gerador consomem posições diferentes da sequência.

In [ ]:
rng_a = np.random.default_rng(SEED)
W_a1 = initialize(rng_a, 8, 5, "he_normal")
W_a2 = initialize(rng_a, 8, 5, "he_normal")
W_b1 = initialize(np.random.default_rng(SEED), 8, 5, "he_normal")

assert np.array_equal(W_a1, W_b1)
assert not np.array_equal(W_a1, W_a2)
assert np.unique(W_a1).size == W_a1.size

digest = hashlib.sha256(W_a1.tobytes()).hexdigest()[:16]
print("Reexecução idêntica:", np.array_equal(W_a1, W_b1))
print("Pesos distintos na matriz:", np.unique(W_a1).size)
print("Hash abreviado:", digest)

## 4. Contraprova de simetria

Criamos dois neurônios ocultos com colunas de entrada e linhas de saída idênticas. O forward e o backward lhes atribuem exatamente os mesmos valores. Depois de uma atualização, continuam clones. Em seguida, a inicialização independente quebra essa igualdade.

In [ ]:
def tanh_network_gradients(X, y, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)
    pred = A1 @ W2 + b2
    residual = pred - y
    loss = 0.5 * np.mean(residual**2)
    d_pred = residual / X.shape[0]
    dW2 = A1.T @ d_pred
    db2 = d_pred.sum(axis=0)
    dA1 = d_pred @ W2.T
    dZ1 = dA1 * (1.0 - A1**2)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0)
    return loss, A1, (dW1, db1, dW2, db2)

rng = np.random.default_rng(SEED)
X = rng.normal(size=(32, 3))
y = rng.normal(size=(32, 1))
incoming = rng.normal(size=(3, 1))
outgoing = rng.normal(size=(1, 1))
W1_same = np.repeat(incoming, 2, axis=1)
b1_same = np.zeros(2)
W2_same = np.repeat(outgoing, 2, axis=0)
b2 = np.zeros(1)

loss_same, A_same, grads_same = tanh_network_gradients(
    X, y, W1_same, b1_same, W2_same, b2
)
dW1_same, db1_same, dW2_same, _ = grads_same
assert np.array_equal(A_same[:, 0], A_same[:, 1])
assert np.array_equal(dW1_same[:, 0], dW1_same[:, 1])
assert np.array_equal(db1_same[0:1], db1_same[1:2])
assert np.array_equal(dW2_same[0:1], dW2_same[1:2])

eta = 0.05
W1_after = W1_same - eta * dW1_same
W2_after = W2_same - eta * dW2_same
assert np.array_equal(W1_after[:, 0], W1_after[:, 1])
assert np.array_equal(W2_after[0], W2_after[1])

W1_random = initialize(np.random.default_rng(SEED + 1), 3, 2, "xavier_normal")
W2_random = initialize(np.random.default_rng(SEED + 2), 2, 1, "xavier_normal")
_, A_random, grads_random = tanh_network_gradients(X, y, W1_random, b1_same, W2_random, b2)
assert not np.allclose(A_random[:, 0], A_random[:, 1])
assert not np.allclose(grads_random[0][:, 0], grads_random[0][:, 1])

print(f"Loss do caso simétrico: {loss_same:.6f}")
print("Diferença máxima entre ativações clones:", np.max(np.abs(A_same[:, 0] - A_same[:, 1])))
print("Diferença máxima após atualização:", np.max(np.abs(W1_after[:, 0] - W1_after[:, 1])))

## 5. Instrumentação da rede profunda

Além da variância, registramos o segundo momento $E[A^2]$, a fração de zeros da ReLU e a saturação da `tanh` definida aqui por $|A|>0{,}99$.

In [ ]:
def activation_forward(z, name, alpha=0.1):
    if name == "relu":
        return np.maximum(z, 0.0)
    if name == "leaky_relu":
        return np.where(z >= 0.0, z, alpha * z)
    if name == "tanh":
        return np.tanh(z)
    raise ValueError(name)


def activation_derivative(z, name, alpha=0.1):
    if name == "relu":
        return (z > 0.0).astype(z.dtype)
    if name == "leaky_relu":
        return np.where(z >= 0.0, 1.0, alpha)
    if name == "tanh":
        t = np.tanh(z)
        return 1.0 - t**2
    raise ValueError(name)


def forward_stack(X, widths, scheme, activation, seed, alpha=0.1):
    rng = np.random.default_rng(seed)
    A = X.copy()
    weights, preacts, activations, stats = [], [], [A], []
    for fan_out in widths:
        W = initialize(rng, A.shape[1], fan_out, scheme, alpha=alpha)
        Z = A @ W
        A = activation_forward(Z, activation, alpha)
        weights.append(W)
        preacts.append(Z)
        activations.append(A)
        stats.append({
            "mean": float(A.mean()),
            "var": float(A.var()),
            "second_moment": float(np.mean(A**2)),
            "zero_fraction": float(np.mean(A == 0.0)),
            "saturation": float(np.mean(np.abs(A) > 0.99)),
        })
    return weights, preacts, activations, stats


def backward_stack(weights, preacts, activation, alpha=0.1):
    # O gradiente final tem segundo momento 1 para tornar comparações legíveis.
    G = np.ones_like(preacts[-1])
    moments = []
    for W, Z in zip(reversed(weights), reversed(preacts)):
        G = G * activation_derivative(Z, activation, alpha)
        moments.append(float(np.mean(G**2)))
        G = G @ W.T
    moments.append(float(np.mean(G**2)))
    return list(reversed(moments))

assert activation_forward(np.array([-1.0, 2.0]), "relu").tolist() == [0.0, 2.0]
assert activation_derivative(np.array([-1.0, 2.0]), "leaky_relu", 0.1).tolist() == [0.1, 1.0]

## 6. ReLU em 20 camadas

Usamos largura 128 e lote sintético. Para ReLU, He deve preservar melhor o segundo momento sob as aproximações da derivação; Xavier tende a encolhê-lo. Escalas fixas extremas fornecem contraprovas.

In [ ]:
batch, width, depth = 2048, 128, 20
X_deep = np.random.default_rng(SEED).normal(size=(batch, width))
widths = [width] * depth
relu_schemes = ["small_normal", "xavier_normal", "he_normal", "large_normal"]
relu_runs = {}

for name in relu_schemes:
    result = forward_stack(X_deep, widths, name, "relu", SEED + 10)
    weights, preacts, activations, stats = result
    relu_runs[name] = result
    first = stats[0]["second_moment"]
    last = stats[-1]["second_moment"]
    print(f"{name:16s} E[A²] camada 1={first:.6e}, camada 20={last:.6e}, razão={last/first:.6e}")

he_last = relu_runs["he_normal"][3][-1]["second_moment"]
xavier_last = relu_runs["xavier_normal"][3][-1]["second_moment"]
small_last = relu_runs["small_normal"][3][-1]["second_moment"]
assert 0.05 < he_last < 20.0
assert xavier_last < he_last / 100.0
assert small_last < 1e-30
assert np.isfinite(relu_runs["large_normal"][2][-1]).all()

### Gráfico — escala das ativações ReLU

O eixo vertical logarítmico deixa visíveis quedas e crescimentos de muitas ordens de grandeza.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
layers = np.arange(1, depth + 1)
for name in relu_schemes:
    values = [s["second_moment"] for s in relu_runs[name][3]]
    ax.plot(layers, values, marker="o", markersize=3, label=name)
ax.set_yscale("log")
ax.set_xlabel("Camada")
ax.set_ylabel("Segundo momento E[A²]")
ax.set_title("Propagação de escala em uma MLP ReLU não treinada")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
print("Texto alternativo: linhas em escala log mostram pesos pequenos e Xavier decaindo; He permanece muito mais estável; pesos grandes crescem rapidamente.")

## 7. `tanh`: escala pequena, Xavier e saturação

Para `tanh`, valores absolutos muito grandes ficam perto de 1 e a derivada se aproxima de zero. A saturação é medida em cada camada.

In [ ]:
tanh_schemes = ["small_normal", "xavier_normal", "he_normal", "large_normal"]
tanh_runs = {}
for name in tanh_schemes:
    result = forward_stack(X_deep, widths, name, "tanh", SEED + 20)
    tanh_runs[name] = result
    stats = result[3]
    print(
        f"{name:16s} var final={stats[-1]['var']:.6e}, "
        f"saturação inicial={stats[0]['saturation']:.3%}, final={stats[-1]['saturation']:.3%}"
    )

xavier_tanh_final = tanh_runs["xavier_normal"][3][-1]["var"]
small_tanh_final = tanh_runs["small_normal"][3][-1]["var"]
large_sat_first = tanh_runs["large_normal"][3][0]["saturation"]
assert xavier_tanh_final > small_tanh_final * 1e10
assert large_sat_first > 0.7
assert tanh_runs["xavier_normal"][3][0]["saturation"] < 0.01

## 8. Backward sintético

Propagamos um gradiente controlado da saída à entrada usando os mesmos pesos do forward. O experimento isola a escala inicial; não simula uma loss específica nem prova convergência.

In [ ]:
backward_results = {}
for name in ("xavier_normal", "he_normal"):
    weights, preacts, _, _ = relu_runs[name]
    moments = backward_stack(weights, preacts, "relu")
    backward_results[name] = moments
    print(
        f"ReLU/{name:14s} E[G²] entrada={moments[0]:.6e}, "
        f"saída={moments[-1]:.6e}, razão={moments[0]/moments[-1]:.6e}"
    )

assert backward_results["xavier_normal"][0] < backward_results["he_normal"][0] / 100.0
assert all(np.isfinite(backward_results[name]).all() for name in backward_results)

### Gráfico — escala do gradiente

As posições seguem da entrada para a saída. A diferença entre as curvas mostra que a distribuição afeta os dois sentidos da rede.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
positions = np.arange(depth + 1)
for name, values in backward_results.items():
    ax.plot(positions, values, marker="o", markersize=3, label=name)
ax.set_yscale("log")
ax.set_xlabel("Posição (0 = entrada; 20 = saída)")
ax.set_ylabel("Segundo momento E[G²]")
ax.set_title("Escala do gradiente em uma MLP ReLU não treinada")
ax.grid(True, which="both", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()
print("Texto alternativo: a curva Xavier chega à entrada muito abaixo da curva He, evidenciando contração do gradiente na rede ReLU.")

## 9. Ganho para Leaky ReLU

Com inclinação negativa $\alpha=0{,}1$, usamos $g=\sqrt{2/(1+\alpha^2)}$. Comparamos o segundo momento da primeira camada ao da entrada.

In [ ]:
alpha = 0.1
gain = np.sqrt(2.0 / (1.0 + alpha**2))
_, _, activations_leaky, stats_leaky = forward_stack(
    X_deep, widths, "he_normal", "leaky_relu", SEED + 30, alpha=alpha
)
input_second = float(np.mean(X_deep**2))
first_second = stats_leaky[0]["second_moment"]
ratio = first_second / input_second
print(f"alpha={alpha}, ganho={gain:.9f}")
print(f"E[X²]={input_second:.6f}, E[A1²]={first_second:.6f}, razão={ratio:.6f}")
assert np.isclose(gain, 1.4071950894605838)
assert 0.9 < ratio < 1.1
assert np.isfinite(activations_leaky[-1]).all()

## 10. Robustez a seeds

Uma única seed pode ser atípica. Repetimos a rede ReLU com He e resumimos o segundo momento da última camada.

In [ ]:
seed_values = [SEED + i for i in range(5)]
final_moments = []
for seed in seed_values:
    _, _, _, stats = forward_stack(X_deep, widths, "he_normal", "relu", seed)
    final_moments.append(stats[-1]["second_moment"])

final_moments = np.asarray(final_moments)
print("E[A20²] por seed:", np.round(final_moments, 6))
print(f"média={final_moments.mean():.6f}, desvio={final_moments.std(ddof=1):.6f}")
assert np.all(np.isfinite(final_moments))
assert np.all(final_moments > 0.01)
assert final_moments.max() / final_moments.min() < 100.0

## 11. Contratos finais

Os asserts abaixo consolidam propriedades teóricas, reprodutibilidade, estabilidade numérica e diferenças esperadas entre esquemas.

In [ ]:
audit = {
    "shape da matriz": W_a1.shape == (8, 5),
    "reexecução por seed": np.array_equal(W_a1, W_b1),
    "chamadas sucessivas distintas": not np.array_equal(W_a1, W_a2),
    "pesos internos distintos": np.unique(W_a1).size == W_a1.size,
    "Xavier normal próximo do alvo": abs(observed["xavier_normal"] - 0.001) < 1e-5,
    "Xavier uniforme próximo do alvo": abs(observed["xavier_uniform"] - 0.001) < 1e-5,
    "He normal próximo do alvo": abs(observed["he_normal"] - 0.002) < 2e-5,
    "He uniforme próximo do alvo": abs(observed["he_uniform"] - 0.002) < 2e-5,
    "simetria no forward": np.array_equal(A_same[:, 0], A_same[:, 1]),
    "simetria no backward": np.array_equal(dW1_same[:, 0], dW1_same[:, 1]),
    "simetria após atualização": np.array_equal(W1_after[:, 0], W1_after[:, 1]),
    "aleatoriedade quebra simetria": not np.allclose(A_random[:, 0], A_random[:, 1]),
    "He mantém ReLU mensurável": 0.05 < he_last < 20.0,
    "Xavier contrai ReLU profunda": xavier_last < he_last / 100.0,
    "peso pequeno colapsa": small_last < 1e-30,
    "peso grande satura tanh": large_sat_first > 0.7,
    "Xavier evita saturação inicial tanh": tanh_runs["xavier_normal"][3][0]["saturation"] < 0.01,
    "backward finito": all(np.isfinite(v).all() for v in backward_results.values()),
    "ganho Leaky ReLU": np.isclose(gain, 1.4071950894605838),
    "Leaky ReLU preserva primeiro momento quadrático": 0.9 < ratio < 1.1,
    "cinco seeds finitas": final_moments.size == 5 and np.isfinite(final_moments).all(),
}

for name, passed in audit.items():
    assert passed, name

print(f"Auditoria: {sum(audit.values())}/{len(audit)} contratos aprovados.")

## Conclusões

- Pesos idênticos produzem neurônios clones; a atualização determinística não rompe a simetria.
- Xavier e He, nas formas normal e uniforme, atingiram suas variâncias-alvo dentro da tolerância amostral.
- Em uma pilha ReLU, He preservou muito melhor o segundo momento de ativações e gradientes do que Xavier.
- Escala fixa pequena colapsou o sinal; escala grande saturou `tanh` já na primeira camada.
- O ganho de Leaky ReLU preservou aproximadamente o segundo momento na primeira camada.
- Várias seeds confirmaram a tendência sem alegar invariância perfeita.

Esses resultados descrevem o estado inicial. Na Aula 16, investigaremos formalmente *vanishing* e *exploding gradients* por produtos de Jacobianos e normas por camada.